In [ ]:
%load_ext autoreload
%autoreload 2

# NeuCodec Audio Tokenization Example

This notebook demonstrates:
1. Loading the ESB diagnostic AMI dataset
2. Selecting long audio samples
3. Tokenizing with NeuCodec to discrete tokens
4. Reconstructing audio from tokens
5. Playing original vs reconstructed audio

## Setup and Imports

In [ ]:
import sys
import torch
import numpy as np
from datasets import load_dataset
from IPython.display import Audio, display
import matplotlib.pyplot as plt

# Add our tokenizer module to path
sys.path.append('../src')

# Import our NeuCodec wrapper
from audio_tokenizers import get_tokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

PyTorch version: 2.9.0+cu128
CUDA available: True
Using device: cuda


## Load ESB Diagnostic AMI Dataset

In [3]:
# Load the AMI clean split
print("Loading ESB diagnostic dataset - AMI clean split...")
dataset = load_dataset('esb/diagnostic-dataset', 'ami', split='clean')

print(f"Dataset loaded with {len(dataset)} samples")
print(f"Sample keys: {dataset[0].keys()}")

Loading ESB diagnostic dataset - AMI clean split...
Dataset loaded with 770 samples
Sample keys: dict_keys(['audio', 'ortho_transcript', 'norm_transcript', 'id', 'dataset'])


## Find Long Audio Samples

In [ ]:
# Calculate duration for each sample and find the longest ones
durations = []
for i, sample in enumerate(dataset):
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    duration = len(audio_array) / sr
    durations.append((i, duration, sample['norm_transcript']))

# Sort by duration and get top 5 longest
durations.sort(key=lambda x: x[1], reverse=True)
long_samples = durations[:5]

print("Top 5 longest audio samples:")
for idx, (i, dur, transcript) in enumerate(long_samples):
    print(f"{idx+1}. Sample {i}: {dur:.2f} seconds")
    print(f"   Transcript: {transcript[:100]}..." if len(transcript) > 100 else f"   Transcript: {transcript}")
    print()

Top 5 longest audio samples:
1. Sample 85: 11.77 seconds
   Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c ...

2. Sample 1: 11.70 seconds
   Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would lik...

3. Sample 100: 11.69 seconds
   Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go ...

4. Sample 31: 11.44 seconds
   Transcript: and also we talked about um a location function where maybe you could press a button on the t v and ...

5. Sample 210: 11.33 seconds
   Transcript: the the problem is if you have to go across the building and it adds some overhead every time you wa...



## Initialize NeuCodec Tokenizer

In [5]:
# Initialize the tokenizer
print("Loading NeuCodec tokenizer...")
tokenizer = get_tokenizer('neucodec', device=device)

print(f"Tokenizer loaded: {tokenizer}")
print(f"  Input sample rate: {tokenizer.sample_rate} Hz")
print(f"  Output sample rate: {tokenizer.output_sample_rate} Hz")
print(f"  Codebook size: {tokenizer.codebook_size:,}")
print(f"  Downsample rate: {tokenizer.downsample_rate}x")

Loading NeuCodec tokenizer...


Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu128 for torchao version 0.14.0         Please see GitHub issue #2919 for more info
/iopsstor/scratch/cscs/xyixuan/benchmark-audio-tokenizers/venv/neucodec/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Tokenizer loaded: NeuCodecTokenizer(checkpoint='neuphonic/neucodec', device='cuda', sample_rate=16000)
  Input sample rate: 16000 Hz
  Output sample rate: 24000 Hz
  Codebook size: 65,536
  Downsample rate: 320x


## Tokenize and Reconstruct Audio Samples

In [6]:
# Process the top 3 longest samples
results = []

for idx in range(min(3, len(long_samples))):
    sample_idx, duration, transcript = long_samples[idx]
    sample = dataset[sample_idx]
    
    print(f"\n{'='*60}")
    print(f"Processing Sample {idx+1} (index {sample_idx})")
    print(f"Duration: {duration:.2f} seconds")
    print(f"Transcript: {transcript[:150]}..." if len(transcript) > 150 else f"Transcript: {transcript}")
    
    # Get audio data
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    
    # Convert to tensor
    audio_tensor = torch.from_numpy(audio_array).float()
    
    # Tokenize
    print("\nTokenizing...")
    tokens, encode_info = tokenizer.encode(audio_tensor, sr=sr)
    
    # Show token information
    print(f"Token shape: {tokens.shape}")
    print(f"Number of tokens: {tokens.numel()}")
    print(f"Tokens per second: {tokens.numel() / duration:.1f}")
    
    # Show first 20 discrete token values
    token_values = tokens.squeeze().cpu().numpy()
    print(f"\nFirst 20 discrete token IDs:")
    print(token_values[:20])
    print(f"Token ID range: [{token_values.min()}, {token_values.max()}]")
    print(f"Unique tokens used: {len(np.unique(token_values))}")
    
    # Decode back to audio
    print("\nDecoding tokens back to audio...")
    reconstructed, decode_info = tokenizer.decode(tokens)
    
    print(f"Reconstructed shape: {reconstructed.shape}")
    print(f"Output sample rate: {decode_info['output_sample_rate']} Hz")
    
    # Store results
    results.append({
        'original': audio_array,
        'original_sr': sr,
        'reconstructed': reconstructed.squeeze().cpu().numpy(),
        'reconstructed_sr': decode_info['output_sample_rate'],
        'tokens': token_values,
        'transcript': transcript,
        'duration': duration
    })
    
print(f"\n{'='*60}")
print("Processing complete!")


Processing Sample 1 (index 85)
Duration: 11.77 seconds
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you...

Tokenizing...
Token shape: torch.Size([1, 1, 588])
Number of tokens: 588
Tokens per second: 50.0

First 20 discrete token IDs:
[22739 52535 52538 23856 55588 51794 22946 46469 52066 64067 54116 53897
  7395 47203 63815 23287 54962 58855 61138 52699]
Token ID range: [786, 65526]
Unique tokens used: 572

Decoding tokens back to audio...
Reconstructed shape: torch.Size([1, 1, 282240])
Output sample rate: 24000 Hz

Processing Sample 2 (index 1)
Duration: 11.70 seconds
Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would like to see in a convenient practical

Tokenizing...
Token shape: torch.Size([1, 1, 585])
Number of tokens: 585
Tokens per second: 50.0

First 20 discrete token IDs:
[ 3219 35893 51490 52529 56868 52

## Play Original vs Reconstructed Audio

In [7]:
# Display audio players for comparison
for idx, result in enumerate(results):
    print(f"\n{'='*60}")
    print(f"Sample {idx+1} - Duration: {result['duration']:.2f}s")
    print(f"Transcript: {result['transcript'][:200]}..." if len(result['transcript']) > 200 else f"Transcript: {result['transcript']}")
    print(f"\nTokens used: {len(result['tokens'])} ({len(result['tokens'])/result['duration']:.1f} tokens/sec)")
    
    print("\nOriginal Audio (16 kHz):")
    display(Audio(result['original'], rate=result['original_sr']))
    
    print("\nReconstructed Audio (24 kHz):")
    display(Audio(result['reconstructed'], rate=result['reconstructed_sr']))
    
    # Compression ratio
    original_size = len(result['original']) * 2  # 16-bit audio
    token_size = len(result['tokens']) * 2  # 16-bit tokens
    compression_ratio = original_size / token_size
    print(f"\nCompression ratio: {compression_ratio:.1f}x")


Sample 1 - Duration: 11.77s
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you sort of have like the number come up on the t v l...

Tokens used: 588 (50.0 tokens/sec)

Original Audio (16 kHz):



Reconstructed Audio (24 kHz):



Compression ratio: 320.3x

Sample 2 - Duration: 11.70s
Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would like to see in a convenient practical

Tokens used: 585 (50.0 tokens/sec)

Original Audio (16 kHz):



Reconstructed Audio (24 kHz):



Compression ratio: 320.0x

Sample 3 - Duration: 11.69s
Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go for that if we want

Tokens used: 584 (50.0 tokens/sec)

Original Audio (16 kHz):



Reconstructed Audio (24 kHz):



Compression ratio: 320.3x


## Summary Statistics

In [11]:
# Calculate summary statistics
print("Summary of Tokenization Results:")
print("="*60)

total_duration = sum(r['duration'] for r in results)
total_tokens = sum(len(r['tokens']) for r in results)

print(f"Total audio processed: {total_duration:.2f} seconds")
print(f"Total tokens generated: {total_tokens:,}")
print(f"Average tokens per second: {total_tokens/total_duration:.1f}")
print(f"Average compression ratio: {16000/320:.1f}x")

print(f"\nNeuCodec Properties:")
print(f"  Bitrate: {50 * 16 / 1000:.1f} kbps")
print(f"  Token rate: 50 Hz")
print(f"  Bits per token: 16")
print(f"  Codebook size: 65,536")
print(f"  Input: 16 kHz mono")
print(f"  Output: 24 kHz mono (upsampled)")

Summary of Tokenization Results:
Total audio processed: 35.16 seconds
Total tokens generated: 1,757
Average tokens per second: 50.0
Average compression ratio: 50.0x

NeuCodec Properties:
  Bitrate: 0.8 kbps
  Token rate: 50 Hz
  Bits per token: 16
  Codebook size: 65,536
  Input: 16 kHz mono
  Output: 24 kHz mono (upsampled)
